In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge

In [2]:
# choose a ticker and sort by date
def Day20Return(market_data, ticker, eventDate):
    df = market_data[market_data['Ticker'] == ticker].sort_values('Date').reset_index(drop=True)
    match = df.index[pd.to_datetime(df['Date'], utc = True).dt.strftime('%Y-%m-%d') == pd.to_datetime(eventDate, utc = True).strftime('%Y-%m-%d')]
    if len(match)==0:
        return np.nan
    idx = match[0]

    # check data avlb
    if idx < 90 or (idx+20) > len(df):
        return np.nan

    # gather data from -90 days prior to +20 days post event
    pre_data = df.iloc[idx-90 : idx][['Close', 'Volume', 'NBI']].values
    post_data = df.iloc[idx : idx+20]['Close'].values

    if (pre_data[0] == 0).any():
        return np.nan

    norm = pre_data/pre_data[0]

    if np.isnan(norm).any() or np.isinf(norm).any() or np.isnan(post_data).any():
        return np.nan

    # sliding window 30d input, 20d outcome
    X = [norm[i : i+30].flatten() for i in range (41)]
    y = [norm[i+30 : i+50, 0] for i in range (41)]

    # fit regression
    pred = Ridge().fit(X,y).predict([norm[-30:].flatten()])[0] * pre_data[0,0]
    return (post_data - pred).mean() / pre_data[-1,0]

In [3]:
market_data_df = pd.read_csv("datasets/market_ds.csv")

In [4]:
catalyst_df = pd.read_csv("datasets/catalyst_ds.csv")

In [5]:
catalyst_df['NCAR20'] = [Day20Return(market_data_df, ticker, date) 
                         for ticker, date in zip(catalyst_df['company_ticker'], 
                                                             pd.to_datetime(catalyst_df['date'], utc = True).dt.strftime('%Y-%m-%d'))]
catalyst_df = catalyst_df.dropna(subset=['NCAR20']).reset_index(drop=True)

In [6]:
catalyst_df_filtered = catalyst_df[(catalyst_df['date'] >= '2025-01-01') & (catalyst_df['date'] <= '2026-06-30')]

In [7]:

company_features_df = pd.read_csv("datasets/company_ds.csv")

In [8]:
final_df = pd.merge(
    catalyst_df_filtered,
    company_features_df,
    left_on='company_ticker',
    right_on='ticker',
    how = 'left',
).drop(columns=['ticker'])

In [9]:
final_df.to_csv("datasets/final_ds_NCAR20.csv", index = False)